# Q3: Communities Defying Assumptions (Outlier Analysis)

**Question:** Are there countries/communities that defy expected patterns — e.g., low access but good outcomes, or high access but poor outcomes?

**Sources:** `v_outlier_communities`, `country_summary`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from notebooks._helpers import (
    get_connection, set_plot_style, save_fig,
)

set_plot_style()
con = get_connection()
print('Connected to DuckDB')

## 1. Load Data

In [ ]:
outlier_df = con.execute('SELECT * FROM v_outlier_communities ORDER BY country').fetchdf()
country_df = con.execute('SELECT * FROM country_summary ORDER BY country').fetchdf()

print(f'v_outlier_communities: {len(outlier_df)} countries')
print(f'Outlier types:')
print(outlier_df['outlier_type'].value_counts().to_string())
outlier_df[['country', 'z_access', 'z_mortality', 'z_recovery', 'outlier_type']]

## 2. Z-Score Analysis

In [ ]:
# Most extreme countries
outlier_df['abs_z_mortality'] = outlier_df['z_mortality'].abs()
outlier_df['abs_z_access'] = outlier_df['z_access'].abs()

print('Top 5 countries by |z_mortality|:')
top_mort = outlier_df.nlargest(5, 'abs_z_mortality')[['country', 'z_mortality', 'avg_mortality']]
print(top_mort.to_string(index=False))

print(f'\nMortality range across all 20 countries:')
print(f'  Min: {outlier_df["avg_mortality"].min():.4f}%')
print(f'  Max: {outlier_df["avg_mortality"].max():.4f}%')
print(f'  Spread: {outlier_df["avg_mortality"].max() - outlier_df["avg_mortality"].min():.4f}pp')
print(f'  Despite some high z-scores, absolute differences are < 0.5pp')

## 3. Relaxed Threshold — Reclassify at z > 0.5

In [ ]:
def classify_relaxed(row):
    if row['z_access'] > 0.5 and row['z_mortality'] > 0.5:
        return 'high_access_poor_outcome'
    if row['z_access'] < -0.5 and row['z_recovery'] > 0.5:
        return 'low_access_good_outcome'
    if row['z_access'] < -0.5 and row['z_mortality'] < -0.5:
        return 'low_access_good_outcome'
    return 'expected_pattern'

outlier_df['relaxed_type'] = outlier_df.apply(classify_relaxed, axis=1)
print('Relaxed classification (z > 0.5):')
print(outlier_df['relaxed_type'].value_counts().to_string())
print()
outlier_df[outlier_df['relaxed_type'] != 'expected_pattern'][['country', 'z_access', 'z_mortality', 'relaxed_type']]

## 4. Defiance Index

In [ ]:
# Defiance index = |z_access + z_mortality| — higher means more "defiant"
outlier_df['defiance_index'] = (outlier_df['z_access'].abs() + outlier_df['z_mortality'].abs())
outlier_df_sorted = outlier_df.sort_values('defiance_index', ascending=False)

print('Defiance Index Ranking (all 20 countries):')
print(outlier_df_sorted[['country', 'defiance_index', 'z_access', 'z_mortality',
                          'avg_access', 'avg_mortality']].to_string(index=False))

## 5. Visualization 1 — Quadrant Scatter: z_access vs z_mortality

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

colors = outlier_df['outlier_type'].map({
    'expected_pattern': '#3498db',
    'high_access_poor_outcome': '#e74c3c',
    'low_access_good_outcome': '#2ecc71',
})

ax.scatter(outlier_df['z_access'], outlier_df['z_mortality'],
           s=100, c=colors, edgecolors='white', linewidth=1.5, zorder=5)

# Label every country
for _, row in outlier_df.iterrows():
    ax.annotate(row['country'],
                (row['z_access'], row['z_mortality']),
                textcoords='offset points', xytext=(8, 4),
                fontsize=8)

# Reference lines at +/- 1 SD
for v in [-1, 1]:
    ax.axhline(v, color='gray', linestyle='--', alpha=0.4)
    ax.axvline(v, color='gray', linestyle='--', alpha=0.4)
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)

ax.set_xlabel('z-score (Healthcare Access)')
ax.set_ylabel('z-score (Mortality Rate)')
ax.set_title('Q3: Country Outlier Quadrant Plot')

# Quadrant labels
ax.text(2, 2, 'High access\nHigh mortality', ha='center', fontsize=9, color='gray', alpha=0.7)
ax.text(-2, -2, 'Low access\nLow mortality', ha='center', fontsize=9, color='gray', alpha=0.7)
ax.text(-2, 2, 'Low access\nHigh mortality', ha='center', fontsize=9, color='gray', alpha=0.7)
ax.text(2, -2, 'High access\nLow mortality', ha='center', fontsize=9, color='gray', alpha=0.7)

save_fig(fig, 'q3_quadrant_scatter')
plt.show()

## 6. Visualization 2 — Radar Chart: Top Deviators vs Global Mean

In [ ]:
# Top 2 deviators + global mean
top2 = outlier_df_sorted.head(2)['country'].tolist()
metrics = ['avg_access', 'avg_mortality', 'avg_recovery', 'avg_income', 'avg_education']
labels = ['Access', 'Mortality', 'Recovery', 'Income', 'Education']

# Normalize to 0-1 range for radar
radar_data = outlier_df.set_index('country')[metrics]
radar_norm = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min() + 1e-10)
global_mean = radar_norm.mean()

angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# Global mean
values = global_mean.tolist() + [global_mean.tolist()[0]]
ax.plot(angles, values, 'o-', linewidth=2, label='Global Mean', color='#95a5a6')
ax.fill(angles, values, alpha=0.1, color='#95a5a6')

colors_radar = ['#e74c3c', '#f39c12']
for country, color in zip(top2, colors_radar):
    if country in radar_norm.index:
        vals = radar_norm.loc[country].tolist() + [radar_norm.loc[country].tolist()[0]]
        ax.plot(angles, vals, 'o-', linewidth=2, label=country, color=color)
        ax.fill(angles, vals, alpha=0.1, color=color)

ax.set_thetagrids(np.degrees(angles[:-1]), labels)
ax.set_title('Q3: Top Deviators vs Global Mean (Normalized)', y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
save_fig(fig, 'q3_radar_deviators')
plt.show()

## 7. Visualization 3 — Defiance Index Ranked Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(
    outlier_df_sorted['country'],
    outlier_df_sorted['defiance_index'],
    color='#3498db', edgecolor='white'
)

# Highlight top deviators
for bar in bars[:3]:
    bar.set_color('#e74c3c')

ax.set_xlabel('Defiance Index (|z_access| + |z_mortality|)')
ax.set_title('Q3: Country Defiance Index Ranking')
ax.invert_yaxis()

for bar, val in zip(bars, outlier_df_sorted['defiance_index']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)

save_fig(fig, 'q3_defiance_index_ranking')
plt.show()

## 8. Key Finding

**No true outlier communities exist in this dataset.**

- While some countries (Russia, Turkey) have the highest z-scores, absolute differences in mortality are < 0.5 percentage points
- The defiance index spread is narrow — the most "deviant" country barely exceeds z = 2.5
- At the relaxed z > 0.5 threshold, some countries are reclassified, but their absolute metrics are virtually identical to the global mean
- The radar chart confirms: top deviators overlap almost perfectly with the global mean profile

**Context:** In real-world health data, we would expect clear outliers (e.g., Cuba's high outcomes despite low income). The absence of meaningful outliers further supports the synthetic-data hypothesis.

In [ ]:
con.close()
print('Q3 analysis complete.')